# ZenGuard UEBA Anomaly Detection Platform

This notebook implements the User and Entity Behavior Analytics (UEBA) component of the ZenGuard Framework using the Isolation Forest algorithm. Following the base paper, it introduces a separate section for **Report Generation** with outputs saved to a designated folder.

### Key Features
- **Synthetic Data Generation**: Maps exactly to the 7 features extracted by ZenGuard SIEM.
- **Automated Report Generation**: Outputs reports and visualizations to `implementation result/`.
- **Interactive SOC Dashboard**: Located in a separate Streamlit script (`dashboard.py`). Use `streamlit run dashboard.py` to view.

In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn streamlit

In [ ]:
import os
import numpy as np
import pandas as pd

# Ensure result directory exists as requested
result_dir = "implementation result"
os.makedirs(result_dir, exist_ok=True)

print("Generating Synthetic Data matching ZenGuard Base Paper...")
np.random.seed(42)
n_samples = 15000

# Normal behavior baselines
session_duration = np.random.normal(loc=2.0, scale=1.5, size=n_samples).clip(0.1, 10)
failed_logins = np.random.poisson(lam=0.2, size=n_samples).clip(0, 3)
access_time = np.random.randint(8, 19, size=n_samples) # Business hours mainly
device_trust_score = np.random.normal(loc=0.9, scale=0.1, size=n_samples).clip(0.5, 1.0)
privilege_change_attempted = np.random.choice([0, 1], p=[0.99, 0.01], size=n_samples)
external_connection = np.random.choice([0, 1], p=[0.8, 0.2], size=n_samples)
MFA_bypassed = np.zeros(n_samples)

data = pd.DataFrame({
    'session_duration': session_duration,
    'failed_logins': failed_logins,
    'access_time': access_time,
    'device_trust_score': device_trust_score,
    'privilege_change_attempted': privilege_change_attempted,
    'external_connection': external_connection,
    'MFA_bypassed': MFA_bypassed
})

# Inject anomalies (15% as per paper)
n_anomalies = int(n_samples * 0.15)
anomaly_indices = np.random.choice(n_samples, n_anomalies, replace=False)

data.loc[anomaly_indices, 'failed_logins'] = np.random.randint(4, 15, size=n_anomalies)
data.loc[anomaly_indices, 'access_time'] = np.random.choice([1, 2, 3, 4, 22, 23], size=n_anomalies)
data.loc[anomaly_indices, 'device_trust_score'] = np.random.uniform(0.1, 0.5, size=n_anomalies)
data.loc[anomaly_indices, 'privilege_change_attempted'] = np.random.choice([0, 1], p=[0.2, 0.8], size=n_anomalies)
data.loc[anomaly_indices, 'external_connection'] = 1
data.loc[anomaly_indices, 'MFA_bypassed'] = np.random.choice([0, 1], p=[0.5, 0.5], size=n_anomalies)

csv_path = os.path.join(result_dir, "synthetic_zenguard_logs.csv")
data.to_csv(csv_path, index=False)
print(f"Generated {n_samples} Synthetic Behavioral Logs! Saved to: {csv_path}")

In [ ]:
from sklearn.ensemble import IsolationForest
import joblib

features = ['session_duration', 'failed_logins', 'access_time', 'device_trust_score', 
            'privilege_change_attempted', 'external_connection', 'MFA_bypassed']
X = data[features]

print("Training ZenGuard UEBA Isolation Forest Model...")
ueba_model = IsolationForest(n_estimators=100, contamination=0.15, random_state=42, n_jobs=-1)
ueba_model.fit(X)

model_path = os.path.join(result_dir, "zenguard_ueba_model.pkl")
joblib.dump(ueba_model, model_path)
print(f"Model successfully saved to: {model_path}")

## Report Generation
Generates risk score distributions, anomaly mapping, and a Markdown evaluation report. Outputs are stored in the `implementation result` directory.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("Generating ZenGuard Evaluation Report...")
data['anomaly_prediction'] = ueba_model.predict(X)
data['anomaly_score'] = ueba_model.decision_function(X)

# Map predictions: -1 (anomaly) -> Risk 95, 1 (normal) -> Risk 50 (From Paper)
data['risk_score'] = np.where(data['anomaly_prediction'] == -1, 95, 50)

plt.figure(figsize=(10, 5))
sns.histplot(data['risk_score'], bins=10, kde=False, color='#2c3e50')
plt.title('UEBA Risk Score Distribution')
plt.xlabel('Risk Score (50=Normal, 95=High Risk)')
plt.ylabel('Frequency')
plt.grid(True, linestyle='--', alpha=0.6)
plot_path = os.path.join(result_dir, 'risk_score_distribution.png')
plt.savefig(plot_path)
plt.close()

plt.figure(figsize=(10, 5))
sns.scatterplot(x='failed_logins', y='anomaly_score', hue='risk_score', 
                palette={50: '#27ae60', 95: '#c0392b'}, data=data, alpha=0.5)
plt.title('Anomaly Score vs Failed Logins')
plt.xlabel('Failed Logins')
plt.ylabel('Isolation Forest Decision Function Score')
plt.grid(True, linestyle='--', alpha=0.6)
scatter_path = os.path.join(result_dir, 'anomaly_scatter.png')
plt.savefig(scatter_path)
plt.close()

report_content = f"""# ZenGuard UEBA Model Evaluation Report\n
## Overview\n- **Total Samples:** {n_samples}\n- **Assumed Contamination Rate:** 15%\n- **High-Risk Events Detected (Risk = 95):** {len(data[data['risk_score'] == 95])}\n- **Normal Events (Risk = 50):** {len(data[data['risk_score'] == 50])}\n
## Model Configuration\n- **Algorithm:** Isolation Forest\n- **Estimators:** 100\n- **Features Analyzed:** {', '.join(features)}\n
## Visualization Artifacts\n- Risk Score Distribution: `risk_score_distribution.png`\n- Anomaly Behavior Scatter Mapping: `anomaly_scatter.png`\n"""
report_path = os.path.join(result_dir, "ueba_evaluation_report.md")
with open(report_path, "w") as f:
    f.write(report_content)
print(f"Reports and plots successfully saved to '{result_dir}' directory.")

## Interactive Streamlit Dashboard
The inline dashboard has been moved to a separate file, as requested.

To run the interactive SOC dashboard that takes manual inputs or SIEM JSON:
1. Open a terminal.
2. Navigate to this directory.
3. Run: `streamlit run dashboard.py`